# Cheap-Talk Signaling — Results Analysis
**ESADE MIBA Capstone 2026 · Strategic Coherence in LLMs**

All numbers are computed from raw data — no hardcoded conclusions.  
Every estimate includes a 95% confidence interval.

**Game conditions**: `aligned` (sender/receiver interests aligned) · `misaligned` (sender incentivised to deceive)  
**Identity conditions**: `undisclosed` · `ai_vs_ai` · `ai_vs_human_blind` · `ai_vs_human_informed` · `human_prior` · `human_vs_human_declared` · `human_vs_human_silent`  
**Key parameters**:
- η (eta) — sender honesty rate
- Δη — honesty drop under misalignment (η_aligned − η_misaligned)
- γ (gamma) — receiver gullibility under misalignment
- β — belief calibration (MAE)

---
### Sections
1. Setup & Data Loading
2. Data Quality Check
3. Sender Honesty η by Game Condition (with 95% CI)
4. Honesty Drop Δη — Incentive-Contingent Deception
5. Receiver Gullibility γ under Misalignment
6. Per-Model η and γ
7. Identity Condition Effect
8. Round-by-Round Dynamics
9. Belief Calibration β
10. Statistical Tests
11. Key Findings Summary

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, pearsonr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR  = os.path.join(REPO_ROOT, 'all_raw')
print(f'Data dir : {DATA_DIR}')
print(f'Exists   : {os.path.isdir(DATA_DIR)}')

In [ ]:
ct_files = sorted(glob.glob(os.path.join(DATA_DIR, 'cheap_talk_results*.csv')))
print(f'Found {len(ct_files)} Cheap-Talk CSV files')

frames = []
for f in ct_files:
    df_tmp = pd.read_csv(f)
    df_tmp.columns = df_tmp.columns.str.strip()
    df_tmp['source_file'] = os.path.basename(f)
    frames.append(df_tmp)

raw = pd.concat(frames, ignore_index=True)
print(f'Total rows          : {len(raw):,}')
print(f'Game conditions     : {sorted(raw["game_condition"].unique())}')
print(f'Identity conditions : {sorted(raw["identity_condition"].unique())}')
print(f'Role rotation       : {sorted(raw["role_rotation"].unique())}')
raw.head(3)

## 2. Data Quality Check

In [ ]:
required = ['run_id','game_condition','identity_condition','role_rotation','matchup','round',
            'true_state','model_sender','message_sent','message_truthful',
            'sender_belief','model_receiver','action_taken','action_correct',
            'receiver_belief','deception_success']
missing = [c for c in required if c not in raw.columns]
assert not missing, f'Missing columns: {missing}'
print('Schema OK')

# Valid values
print(f'true_state values      : {sorted(raw["true_state"].unique())}')
print(f'message_sent values    : {sorted(raw["message_sent"].unique())}')
print(f'action_taken values    : {sorted(raw["action_taken"].unique())}')
print(f'message_truthful range : {raw["message_truthful"].unique()}')
print(f'action_correct range   : {raw["action_correct"].unique()}')

# Belief ranges
for col in ['sender_belief','receiver_belief']:
    out = raw[(raw[col] < 0) | (raw[col] > 1)]
    print(f'{col} out of [0,1] : {len(out)} rows')

print('\nRows per game_condition × identity_condition:')
print(raw.groupby(['game_condition','identity_condition']).size().to_string())

In [ ]:
# Ensure boolean columns are numeric
for col in ['message_truthful','action_correct','deception_success']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

def mean_ci(x, confidence=0.95):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    m = x.mean()
    if set(np.unique(x)).issubset({0.0, 1.0}):
        from statsmodels.stats.proportion import proportion_confint
        lo, hi = proportion_confint(int(x.sum()), n, alpha=1-confidence, method='wilson')
        return m, lo, hi
    se = stats.sem(x)
    t  = stats.t.ppf((1 + confidence) / 2, df=n-1)
    return m, m - t*se, m + t*se

def mcolor(name):
    n = name.lower()
    if 'claude'  in n: return '#8b5cf6'
    if 'gpt'     in n: return '#10b981'
    if 'gemini'  in n: return '#60a5fa'
    if 'human'   in n: return '#e8865a'
    return '#888'

print(f'Senders   : {sorted(raw["model_sender"].unique())}')
print(f'Receivers : {sorted(raw["model_receiver"].unique())}')

## 3. Sender Honesty η by Game Condition

In [ ]:
# Game-level aggregation: one η per game (run × sender)
game_level = (
    raw.groupby(['source_file','run_id','game_condition','identity_condition',
                 'role_rotation','matchup','model_sender','model_receiver'])
    .agg(
        eta          = ('message_truthful', 'mean'),
        action_acc   = ('action_correct',   'mean'),
        deception    = ('deception_success','mean'),
        sender_beta  = ('sender_belief',    lambda x: (x - raw.loc[x.index,'message_truthful']).abs().mean()),
        receiver_beta= ('receiver_belief',  lambda x: (x - raw.loc[x.index,'message_truthful']).abs().mean()),
        n_rounds     = ('round', 'count'),
    ).reset_index()
)

honesty_stats = []
for cond, grp in game_level.groupby('game_condition'):
    m, lo, hi = mean_ci(grp['eta'])
    honesty_stats.append({'game_condition':cond,'eta':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
honesty_df = pd.DataFrame(honesty_stats)

print('=== η (Sender Honesty) by Game Condition ===')
for _, r in honesty_df.iterrows():
    print(f"  {r['game_condition']:12s}  η={r['eta']:.3f}  95% CI [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]  (n={r['n']} games)")

In [ ]:
gc_colors = {'aligned':'#52c97a','misaligned':'#e8865a'}
game_conds = sorted(honesty_df['game_condition'].unique())

fig, ax = plt.subplots(figsize=(7, 5))
for i, (_, r) in enumerate(honesty_df.iterrows()):
    ax.bar(i, r['eta'], color=gc_colors.get(r['game_condition'],'#aaa'), alpha=0.85, width=0.5)
    ax.errorbar(i, r['eta'],
                yerr=[[r['eta']-r['ci_lo']], [r['ci_hi']-r['eta']]],
                fmt='none', color='black', capsize=7, linewidth=1.8)
    ax.text(i, r['eta'] + 0.02, f"{r['eta']:.1%}",
            ha='center', fontsize=12, fontweight='bold')

ax.set_xticks(range(len(honesty_df)))
ax.set_xticklabels(honesty_df['game_condition'].tolist())
ax.set_ylabel('η (Sender Honesty Rate)')
ax.set_ylim(0, 1.1)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5, label='50% random')
ax.set_title('Sender Honesty by Game Condition\n(error bars = 95% Wilson CI)')
ax.legend()
plt.tight_layout()
plt.savefig('../notebooks/fig_ct_honesty_condition.png', bbox_inches='tight')
plt.show()

## 4. Honesty Drop Δη — Incentive-Contingent Deception

In [ ]:
# Δη = η_aligned − η_misaligned, computed per sender model
delta_rows = []
for sender, grp in game_level.groupby('model_sender'):
    aligned_eta    = grp[grp['game_condition']=='aligned']['eta'].dropna()
    misaligned_eta = grp[grp['game_condition']=='misaligned']['eta'].dropna()
    if len(aligned_eta) == 0 or len(misaligned_eta) == 0:
        continue
    m_a, lo_a, hi_a = mean_ci(aligned_eta)
    m_m, lo_m, hi_m = mean_ci(misaligned_eta)
    delta = m_a - m_m
    # Bootstrap CI for Δη
    rng = np.random.default_rng(42)
    deltas = []
    for _ in range(1000):
        s_a = rng.choice(aligned_eta.values,    size=len(aligned_eta),    replace=True).mean()
        s_m = rng.choice(misaligned_eta.values, size=len(misaligned_eta), replace=True).mean()
        deltas.append(s_a - s_m)
    lo_d, hi_d = np.percentile(deltas, [2.5, 97.5])
    delta_rows.append({
        'model_sender': sender,
        'eta_aligned':  m_a, 'eta_misaligned': m_m,
        'delta_eta':    delta, 'ci_lo': lo_d, 'ci_hi': hi_d,
        'n_aligned': len(aligned_eta), 'n_misaligned': len(misaligned_eta)
    })
delta_df = pd.DataFrame(delta_rows).sort_values('delta_eta', ascending=False)

print('=== Δη (Honesty Drop under Misalignment) — positive = more deception ===')
for _, r in delta_df.iterrows():
    sig = '*' if r['ci_lo'] > 0 else ''
    print(f"  {r['model_sender']:35s}  "
          f"η_aligned={r['eta_aligned']:.3f}  η_misaligned={r['eta_misaligned']:.3f}  "
          f"Δη={r['delta_eta']:+.3f}  95% CI [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]  {sig}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: η by game condition per sender model
ax = axes[0]
senders = delta_df['model_sender'].tolist()
x = np.arange(len(senders))
width = 0.35
for j, (cond, color) in enumerate([('aligned','#52c97a'),('misaligned','#e8865a')]):
    vals = delta_df[f'eta_{cond}'].values
    ax.bar(x + (j-0.5)*width, vals, width, color=color, alpha=0.85, label=cond)
ax.set_xticks(x)
ax.set_xticklabels(senders, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('η (Honesty Rate)')
ax.set_ylim(0, 1.05)
ax.set_title('η by Game Condition per Sender Model')
ax.legend(title='Game condition')

# Right: Δη with bootstrap CI
ax = axes[1]
colors_delta = ['#52c97a' if r['ci_lo'] > 0 else '#94a3b8' for _, r in delta_df.iterrows()]
ax.barh(senders, delta_df['delta_eta'].values, color=colors_delta, alpha=0.85)
ax.errorbar(delta_df['delta_eta'].values, senders,
            xerr=[(delta_df['delta_eta']-delta_df['ci_lo']).values,
                  (delta_df['ci_hi']-delta_df['delta_eta']).values],
            fmt='none', color='#333', capsize=5, linewidth=1.3)
ax.axvline(0, color='black', linewidth=0.9)
ax.set_xlabel('Δη = η_aligned − η_misaligned')
ax.set_title('Honesty Drop (Δη) per Sender Model\n(green = CI excludes 0, bootstrap)')

plt.tight_layout()
plt.savefig('../notebooks/fig_ct_delta_eta.png', bbox_inches='tight')
plt.show()

## 5. Receiver Gullibility γ under Misalignment

In [ ]:
# γ = Pearson correlation between message_truthful and action_correct on misaligned runs
# High γ = receiver follows messages even when sender has incentive to deceive
mis = raw[raw['game_condition']=='misaligned'].copy()

gamma_rows = []
for receiver, grp in mis.groupby('model_receiver'):
    valid = grp[['message_truthful','action_correct']].dropna()
    if len(valid) < 10:
        continue
    r, pval = pearsonr(valid['message_truthful'], valid['action_correct'])
    # Bootstrap CI for r
    rng = np.random.default_rng(42)
    boot_r = []
    for _ in range(1000):
        idx = rng.integers(0, len(valid), size=len(valid))
        s = valid.iloc[idx]
        if s['message_truthful'].std() == 0 or s['action_correct'].std() == 0:
            continue
        boot_r.append(pearsonr(s['message_truthful'], s['action_correct'])[0])
    lo, hi = np.percentile(boot_r, [2.5, 97.5]) if boot_r else (np.nan, np.nan)
    gamma_rows.append({
        'model_receiver': receiver,
        'gamma': r, 'ci_lo': lo, 'ci_hi': hi,
        'p_value': pval, 'n': len(valid)
    })
gamma_df = pd.DataFrame(gamma_rows).sort_values('gamma', ascending=False)

print('=== γ (Receiver Gullibility — misaligned runs only) ===')
print('γ > 0 means receiver follows sender messages (gullible under deception incentive)')
for _, r in gamma_df.iterrows():
    sig = '*' if r['p_value'] < 0.05 else ''
    print(f"  {r['model_receiver']:35s}  γ={r['gamma']:+.3f}  "
          f"95% CI [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]  p={r['p_value']:.4f}  {sig}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
receivers = gamma_df['model_receiver'].tolist()
colors_g = ['#e8865a' if r['ci_lo'] > 0 else '#94a3b8' for _, r in gamma_df.iterrows()]
ax.barh(receivers, gamma_df['gamma'].values, color=colors_g, alpha=0.85)
ax.errorbar(gamma_df['gamma'].values, receivers,
            xerr=[(gamma_df['gamma']-gamma_df['ci_lo']).values,
                  (gamma_df['ci_hi']-gamma_df['gamma']).values],
            fmt='none', color='#333', capsize=5, linewidth=1.3)
ax.axvline(0, color='black', linewidth=0.9)
ax.set_xlabel('γ (Pearson r: message truthful × action correct, misaligned only)')
ax.set_title('Receiver Gullibility γ under Misalignment\n(orange = CI excludes 0, bootstrap)')
plt.tight_layout()
plt.savefig('../notebooks/fig_ct_gamma.png', bbox_inches='tight')
plt.show()

## 6. Per-Model η and γ Overview

In [ ]:
# Combined sender/receiver view per model
all_models = sorted(set(raw['model_sender'].tolist() + raw['model_receiver'].tolist()))

model_summary = []
for model in all_models:
    as_sender   = game_level[game_level['model_sender']   == model]
    as_receiver = game_level[game_level['model_receiver'] == model]
    eta_all,  lo_e, hi_e = mean_ci(as_sender['eta'].dropna()) if len(as_sender) else (np.nan,)*3
    acc_all,  lo_a, hi_a = mean_ci(as_receiver['action_acc'].dropna()) if len(as_receiver) else (np.nan,)*3
    dec_all,  lo_d, hi_d = mean_ci(as_sender[as_sender['game_condition']=='misaligned']['deception'].dropna()) \
                           if len(as_sender) else (np.nan,)*3
    gamma_val = gamma_df.set_index('model_receiver').reindex([model])['gamma'].values[0] \
                if model in gamma_df['model_receiver'].values else np.nan
    model_summary.append({
        'model': model,
        'eta': eta_all, 'eta_lo': lo_e, 'eta_hi': hi_e,
        'receiver_acc': acc_all, 'acc_lo': lo_a, 'acc_hi': hi_a,
        'deception_success': dec_all,
        'gamma': gamma_val,
    })
model_df = pd.DataFrame(model_summary)

print('=== Per-Model Summary ===')
print(model_df[['model','eta','receiver_acc','deception_success','gamma']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models_list = model_df['model'].tolist()
colors_list = [mcolor(m) for m in models_list]

# Left: η (sender honesty)
ax = axes[0]
ax.barh(models_list, model_df['eta'].values, color=colors_list, alpha=0.82)
ax.errorbar(model_df['eta'].values, models_list,
            xerr=[(model_df['eta']-model_df['eta_lo']).values,
                  (model_df['eta_hi']-model_df['eta']).values],
            fmt='none', color='#333', capsize=4, linewidth=1.3)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5, label='50% random')
ax.set_xlabel('η (Honesty Rate as Sender)')
ax.set_title('Sender Honesty η by Model\n(all conditions, 95% CI)')

# Right: receiver accuracy
ax = axes[1]
ax.barh(models_list, model_df['receiver_acc'].fillna(0).values, color=colors_list, alpha=0.82)
ax.errorbar(model_df['receiver_acc'].fillna(0).values, models_list,
            xerr=[(model_df['receiver_acc']-model_df['acc_lo']).fillna(0).values,
                  (model_df['acc_hi']-model_df['receiver_acc']).fillna(0).values],
            fmt='none', color='#333', capsize=4, linewidth=1.3)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5, label='50% random')
ax.set_xlabel('Action Accuracy as Receiver')
ax.set_title('Receiver Accuracy by Model\n(all conditions, 95% CI)')

plt.tight_layout()
plt.savefig('../notebooks/fig_ct_model_overview.png', bbox_inches='tight')
plt.show()

## 7. Identity Condition Effect

In [ ]:
identity_stats = []
for (ident, game_cond), grp in game_level.groupby(['identity_condition','game_condition']):
    m_eta, lo_e, hi_e = mean_ci(grp['eta'])
    m_acc, lo_a, hi_a = mean_ci(grp['action_acc'])
    identity_stats.append({
        'identity': ident, 'game_condition': game_cond,
        'eta': m_eta, 'eta_lo': lo_e, 'eta_hi': hi_e,
        'acc': m_acc, 'acc_lo': lo_a, 'acc_hi': hi_a,
        'n': len(grp)
    })
identity_df = pd.DataFrame(identity_stats)

print('=== η by Identity × Game Condition ===')
for _, r in identity_df.sort_values(['game_condition','identity']).iterrows():
    print(f"  [{r['game_condition']:10s}] {r['identity']:30s}  "
          f"η={r['eta']:.3f} [{r['eta_lo']:.3f},{r['eta_hi']:.3f}]  "
          f"acc={r['acc']:.3f}  n={r['n']}")

In [ ]:
game_cond_vals = sorted(identity_df['game_condition'].unique())
ident_vals = sorted(identity_df['identity'].unique())
x = np.arange(len(ident_vals))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
for j, (gc, color) in enumerate([('aligned','#52c97a'),('misaligned','#e8865a')]):
    sub = identity_df[identity_df['game_condition']==gc].set_index('identity').reindex(ident_vals)
    offset = (j - 0.5) * width
    ax.bar(x + offset, sub['eta'].fillna(0), width, color=color, alpha=0.85, label=gc)
    ax.errorbar(x + offset, sub['eta'].fillna(0),
                yerr=[(sub['eta']-sub['eta_lo']).fillna(0),
                      (sub['eta_hi']-sub['eta']).fillna(0)],
                fmt='none', color='#222', capsize=4, linewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels(ident_vals, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('η (Sender Honesty Rate)')
ax.set_ylim(0, 1.1)
ax.set_title('Sender Honesty by Identity × Game Condition\n(error bars = 95% Wilson CI)')
ax.legend(title='Game condition')
plt.tight_layout()
plt.savefig('../notebooks/fig_ct_identity_effect.png', bbox_inches='tight')
plt.show()

## 8. Round-by-Round Dynamics

In [ ]:
round_stats = []
for (gc, rnd), grp in raw.groupby(['game_condition','round']):
    m_h, lo_h, hi_h = mean_ci(grp['message_truthful'])
    m_a, lo_a, hi_a = mean_ci(grp['action_correct'])
    round_stats.append({'game_condition':gc,'round':rnd,
                        'honesty':m_h,'h_lo':lo_h,'h_hi':hi_h,
                        'accuracy':m_a,'a_lo':lo_a,'a_hi':hi_a})
round_df = pd.DataFrame(round_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for gc, color in [('aligned','#52c97a'),('misaligned','#e8865a')]:
    s = round_df[round_df['game_condition']==gc].sort_values('round')
    axes[0].plot(s['round'], s['honesty'], label=gc, color=color, linewidth=1.8, marker='o', markersize=3)
    axes[0].fill_between(s['round'], s['h_lo'], s['h_hi'], color=color, alpha=0.12)
    axes[1].plot(s['round'], s['accuracy'], label=gc, color=color, linewidth=1.8, marker='o', markersize=3)
    axes[1].fill_between(s['round'], s['a_lo'], s['a_hi'], color=color, alpha=0.12)

for ax, title, ylabel in zip(axes,
    ['Sender Honesty Over Rounds', 'Receiver Accuracy Over Rounds'],
    ['η (Honesty Rate)', 'Action Accuracy']):
    ax.set_xlabel('Round')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
    ax.set_title(f'{title}\n(shaded = 95% CI)')
    ax.legend(title='Game condition')
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('../notebooks/fig_ct_round_dynamics.png', bbox_inches='tight')
plt.show()

## 9. Belief Calibration β

In [ ]:
# Sender β: |sender_belief - message_truthful| (how well does sender predict receiver follow-through?)
# Receiver β: |receiver_belief - message_truthful| (how well calibrated is receiver's trust?)
raw['sender_beta_err']   = (raw['sender_belief']   - raw['message_truthful']).abs()
raw['receiver_beta_err'] = (raw['receiver_belief'] - raw['message_truthful']).abs()

beta_rows = []
for (sender, gc), grp in raw.groupby(['model_sender','game_condition']):
    m, lo, hi = mean_ci(grp['sender_beta_err'].dropna())
    beta_rows.append({'model':sender,'role':'sender','game_condition':gc,
                      'beta':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
for (receiver, gc), grp in raw.groupby(['model_receiver','game_condition']):
    m, lo, hi = mean_ci(grp['receiver_beta_err'].dropna())
    beta_rows.append({'model':receiver,'role':'receiver','game_condition':gc,
                      'beta':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
beta_df = pd.DataFrame(beta_rows)

print('=== β (Belief Calibration MAE) by Role and Game Condition ===')
for _, r in beta_df.sort_values(['role','model','game_condition']).iterrows():
    print(f"  {r['role']:8s} {r['model']:35s} [{r['game_condition']:10s}]  "
          f"β={r['beta']:.3f}  95% CI [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]")

## 10. Statistical Tests

In [ ]:
print('=== Mann-Whitney U: η aligned vs misaligned (H2 test) ===')
eta_a = game_level[game_level['game_condition']=='aligned']['eta'].dropna()
eta_m = game_level[game_level['game_condition']=='misaligned']['eta'].dropna()
if len(eta_a) >= 3 and len(eta_m) >= 3:
    stat, p = mannwhitneyu(eta_a, eta_m, alternative='two-sided')
    print(f'  U={stat:.0f},  p={p:.4f}')
    print(f'  η_aligned={eta_a.mean():.3f},  η_misaligned={eta_m.mean():.3f},  Δη={eta_a.mean()-eta_m.mean():+.3f}')

print('\n=== Kruskal-Wallis: η across identity conditions ===')
groups = [grp['eta'].dropna().values
          for _, grp in game_level.groupby('identity_condition') if len(grp) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f'  H={stat:.3f},  p={p:.4f}')

print('\n=== Kruskal-Wallis: η across sender models ===')
groups = [grp['eta'].dropna().values
          for _, grp in game_level.groupby('model_sender') if len(grp) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f'  H={stat:.3f},  p={p:.4f}')

In [ ]:
print('=== Pairwise Mann-Whitney U (models, Holm-Bonferroni corrected) ===')
sender_models = sorted(game_level['model_sender'].unique())
pairs = list(itertools.combinations(sender_models, 2))
raw_p, labels = [], []
for m1, m2 in pairs:
    g1 = game_level[game_level['model_sender']==m1]['eta'].dropna()
    g2 = game_level[game_level['model_sender']==m2]['eta'].dropna()
    if len(g1) < 3 or len(g2) < 3:
        continue
    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
    raw_p.append(p)
    labels.append(f'{m1} vs {m2}')
if raw_p:
    reject, p_adj, _, _ = multipletests(raw_p, method='holm')
    for label, p_raw, p_c, rej in zip(labels, raw_p, p_adj, reject):
        sig = '***' if p_c<0.001 else '**' if p_c<0.01 else '*' if p_c<0.05 else 'ns'
        print(f'  {label:55s}  p_raw={p_raw:.4f}  p_adj={p_c:.4f}  {sig}')

## 11. Key Findings Summary

In [ ]:
print('=' * 65)
print('KEY FINDINGS — Cheap-Talk Signaling')
print('=' * 65)

print('\n[DATASET]')
print(f'  Total runs          : {len(ct_files)}')
print(f'  Total rounds        : {len(raw):,}')
print(f'  Game conditions     : {", ".join(sorted(raw["game_condition"].unique()))}')
print(f'  Identity conditions : {", ".join(sorted(raw["identity_condition"].unique()))}')
print(f'  Sender models       : {", ".join(sorted(raw["model_sender"].unique()))}')

print('\n[η BY GAME CONDITION]')
for _, r in honesty_df.iterrows():
    print(f'  {r["game_condition"]:12s}  η={r["eta"]:.3f}  95% CI [{r["ci_lo"]:.3f}, {r["ci_hi"]:.3f}]  (n={r["n"]} games)')

print('\n[Δη BY SENDER MODEL (H2: incentive-contingent deception)]')
for _, r in delta_df.iterrows():
    sig = '* CI excludes 0' if r['ci_lo'] > 0 else ''
    print(f'  {r["model_sender"]:35s}  Δη={r["delta_eta"]:+.3f}  [{r["ci_lo"]:+.3f}, {r["ci_hi"]:+.3f}]  {sig}')

print('\n[γ RECEIVER GULLIBILITY (misaligned only)]')
for _, r in gamma_df.iterrows():
    sig = '* p<0.05' if r['p_value'] < 0.05 else ''
    print(f'  {r["model_receiver"]:35s}  γ={r["gamma"]:+.3f}  [{r["ci_lo"]:+.3f}, {r["ci_hi"]:+.3f}]  {sig}')

print('\n[NOTE] All values computed from raw CSVs.')
print('       η CIs: Wilson. Δη CIs: bootstrap (n=1000). γ CIs: bootstrap (n=1000).')
print('       Tests: Mann-Whitney U (H2), Kruskal-Wallis + Holm-Bonferroni (H1, H3).')